In [5]:
import ast, json

# Show an example of what one JSON record will look like
example = train.iloc[0].copy()

# case_name is stored as a string representation of a list — parse it
example["case_name"] = ast.literal_eval(example["case_name"])

# overruling_case may be NaN
if pd.isna(example.get("overruling_case")):
    example["overruling_case"] = None

print(json.dumps(example.to_dict(), indent=2))


{
  "claim": "In capital cases involving interracial crimes, defendants have a right to inform potential jurors of the victim's race and to inquire about potential racial bias.",
  "case_name": [
    "Turner v. Murray"
  ],
  "overruling_case": null,
  "label": 0,
  "class": "Refuted"
}


In [ ]:
def _parse_case_name(value):
    """Parse case_name robustly into a list of strings."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []

        # First try JSON parsing
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except Exception:
            pass

        # Then try Python literal parsing
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except (ValueError, SyntaxError):
            # Final fallback for malformed bracket/comma strings
            cleaned = text.strip("[]").strip()
            if not cleaned:
                return []
            parts = [p.strip().strip('"').strip("'") for p in cleaned.split(",")]
            return [p for p in parts if p]

    return [str(value)]


def df_to_json_records(df):
    """Convert a dataset DataFrame to a list of clean JSON records."""
    records = []
    for _, row in df.iterrows():
        record = {
            "claim": row["claim"],
            "case_name": _parse_case_name(row.get("case_name")),
            "overruling_case": row["overruling_case"] if pd.notna(row.get("overruling_case")) else None,
            "label": row["class"],
        }
        records.append(record)
    return records

train_records = df_to_json_records(train)
test_records = df_to_json_records(test)

with open(f"{dataset_dir}/train_set.json", "w", encoding="utf-8") as f:
    json.dump(train_records, f, indent=2, ensure_ascii=False)

with open(f"{dataset_dir}/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_records, f, indent=2, ensure_ascii=False)

print(f"Saved train_set.json ({len(train_records)} records)")
print(f"Saved test_set.json  ({len(test_records)} records)")


Saved train_set.json (5794 records)
Saved test_set.json  (500 records)
